# Tutorial 1: Introduction to ACT-R and pyactr

ACT-R (Adaptive Control of Thought—Rational) is a cognitive architecture for simulating human cognition. This tutorial introduces the core components and shows you how to build your first model using pyactr.

## Table of Contents
1. [What is ACT-R?](#what-is-actr)
2. [Core Concepts](#core-concepts)
3. [Your First pyactr Model](#first-model)
4. [Understanding Model Behavior](#understanding-behavior)
5. [Exercises](#exercises)

## What is ACT-R? <a id='what-is-actr'></a>

ACT-R is a computational theory of mind that lets you simulate human cognition. You can use it to model human behavior in tasks, test psychological theories by implementing them, and make predictions about how people will behave in new situations.

Traditional psychology experiments tell us what people do. ACT-R helps us understand why they do it by forcing us to specify the exact cognitive mechanisms involved.

You'll see ACT-R applied in education (modeling how people learn skills), interface design (predicting user errors), and clinical psychology (modeling cognitive deficits and interventions).

## Core Concepts <a id='core-concepts'></a>

ACT-R has several key components that work together.

**Chunks** are the basic units of knowledge. They represent facts, goals, or any piece of information. For example, a chunk might represent "The capital of France is Paris" with fields for country and capital.

**Buffers** hold chunks that are currently being processed—they're essentially your working memory. The goal buffer holds what you're trying to do, the retrieval buffer holds what you just remembered, and the visual buffer holds what you're looking at.

**Productions** are IF-THEN rules that drive behavior. If certain chunks are in certain buffers, then perform certain actions.

**Modules** are the cognitive subsystems. Declarative memory stores long-term facts, procedural memory stores knowledge of how to do things (the productions), and there are modules for vision and motor control.

## Your First pyactr Model <a id='first-model'></a>

We'll build a simple counting model. This is traditionally the first model because it demonstrates the core concepts clearly.

In [ ]:
import pyactr as actr
import time
import pandas as pd
import matplotlib.pyplot as plt

print(f"pyactr version: {actr.__version__}")

### Step 1: Define Chunk Types

We define the structure of our knowledge. For counting, we need to know what comes after each number (e.g., after 1 comes 2) and what our current counting state is.

In [ ]:
# Counting facts: "first X, second Y"
actr.chunktype("countOrder", ("first", "second"))

# Counting goal: "I'm counting from X to Y, currently at Z"
actr.chunktype("countGoal", ("start", "end", "current"))

print("Chunk types defined!")

### Step 2: Create the Model

In [ ]:
counting_model = actr.ACTRModel()
dm = counting_model.decmem

# Add counting facts to memory
counting_facts = [
    actr.makechunk(typename="countOrder", first=1, second=2),
    actr.makechunk(typename="countOrder", first=2, second=3),
    actr.makechunk(typename="countOrder", first=3, second=4),
    actr.makechunk(typename="countOrder", first=4, second=5),
    actr.makechunk(typename="countOrder", first=5, second=6),
]

for fact in counting_facts:
    dm.add(fact)

print(f"Model created with {len(dm)} facts in memory")
print("\nMemory contents:")
for chunk in dm:
    print(f"  {chunk}")

### Step 3: Define Productions (Rules)

Productions define how the model behaves based on its current state.

In [ ]:
# Start counting
counting_model.productionstring(name="start", string="""
    =g>
        isa     countGoal
        start   =x
        current None
    ==>
    =g>
        current =x
    +retrieval>
        isa     countOrder
        first   =x
""")

# Increment the count
counting_model.productionstring(name="increment", string="""
    =g>
        isa     countGoal
        current =x
        end     ~=x
    =retrieval>
        isa     countOrder
        first   =x
        second  =y
    ==>
    =g>
        current =y
    +retrieval>
        isa     countOrder  
        first   =y
""")

# Stop when we reach the end
counting_model.productionstring(name="stop", string="""
    =g>
        isa     countGoal
        current =x
        end     =x
    ==>
    ~g>
""")

print("Productions defined!")

### Step 4: Run the Model

In [ ]:
counting_model.goal.add(actr.makechunk(typename="countGoal", start=2, end=5))

print("Starting simulation: Count from 2 to 5\n")
print("Time\tProduction\tCurrent\tAction")
print("-" * 60)

simulation_log = []

import io
import sys

old_stdout = sys.stdout
sys.stdout = buffer = io.StringIO()

sim_trace = counting_model.simulation(gui=False, trace=True)
sim_trace.run(10)

trace_output = buffer.getvalue()
sys.stdout = old_stdout

# Parse trace output
for line in trace_output.split('\n'):
    if 'RULE FIRED:' in line:
        parts = line.strip('()').split(', ')
        if len(parts) >= 3:
            time = float(parts[0])
            production = parts[2].split(': ')[1]
            
            log_entry = {
                'time': time,
                'production': production,
                'current': 'N/A',
                'action': f"Production '{production}' fired"
            }
            
            simulation_log.append(log_entry)
            print(f"{time:.3f}\t{production}\t\tN/A\tProduction '{production}' fired")

print("\nSimulation complete!")

## Understanding Model Behavior <a id='understanding-behavior'></a>

Now we can visualize and analyze what happened during the simulation.

In [ ]:
if simulation_log:
    df = pd.DataFrame(simulation_log)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(df['time'], df['production'], s=100, alpha=0.6)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Production')
    plt.title('Production Firing Timeline')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nTiming Analysis:")
    print(f"Total simulation time: {df['time'].max():.3f} seconds")
    print(f"Number of productions fired: {len(df)}")
    if len(df) > 1:
        time_diffs = df['time'].diff().dropna()
        print(f"Average time between productions: {time_diffs.mean():.3f} seconds")
        print(f"Standard deviation: {time_diffs.std():.3f} seconds")
else:
    print("No production firings were logged. Check that the simulation ran correctly.")

Notice that productions don't fire instantly—ACT-R includes realistic cognitive timing. Only one production fires at a time, modeling human attention limitations. Memory retrieval takes time.

What if we change the timing parameters?

In [ ]:
fast_model = actr.ACTRModel()

# Make retrieval faster (default latency factor is 0.1)
fast_model.model_parameters["latency_factor"] = 0.02
fast_model.model_parameters["latency_exponent"] = 0.5

dm = fast_model.decmem
for fact in counting_facts:
    dm.add(fact)

# Define productions
fast_model.productionstring(name="start", string="""
    =g>
        isa     countGoal
        start   =x
        current None
    ==>
    =g>
        current =x
    +retrieval>
        isa     countOrder
        first   =x
""")

fast_model.productionstring(name="increment", string="""
    =g>
        isa     countGoal
        current =x
        end     ~=x
    =retrieval>
        isa     countOrder
        first   =x
        second  =y
    ==>
    =g>
        current =y
    +retrieval>
        isa     countOrder  
        first   =y
""")

fast_model.productionstring(name="stop", string="""
    =g>
        isa     countGoal
        current =x
        end     =x
    ==>
    ~g>
""")

fast_model.goal.add(actr.makechunk(typename="countGoal", start=2, end=5))

import io
import sys

fast_stdout = sys.stdout
sys.stdout = fast_buffer = io.StringIO()

fast_sim = fast_model.simulation(gui=False, trace=True)
fast_sim.run(10)

fast_trace = fast_buffer.getvalue()
sys.stdout = fast_stdout

# Parse fast model trace
fast_times = []
for line in fast_trace.split('\n'):
    if 'RULE FIRED:' in line:
        parts = line.strip('()').split(', ')
        if len(parts) >= 3:
            time = float(parts[0])
            fast_times.append(time)

print("Comparison of model speeds:")
if simulation_log and fast_times:
    original_time = simulation_log[-1]['time'] if simulation_log else 0
    fast_time = fast_times[-1] if fast_times else 0
    
    print(f"\nOriginal model total time: {original_time:.3f} seconds")
    print(f"Fast model total time: {fast_time:.3f} seconds")
    
    if fast_time > 0:
        print(f"\nSpeedup factor: {original_time / fast_time:.1f}x")
    
    print("\nDifferent people have different cognitive speeds, and expertise can lead to faster retrieval.")
    print("This allows us to model individual differences.")
else:
    print("Unable to compare - check that both simulations ran successfully")

## Exercises <a id='exercises'></a>

Try these exercises to deepen your understanding.

### Exercise 1: Extend the Counting Range

Modify the model to count from 1 to 10.

In [ ]:
# Your code here

# extended_model = actr.ACTRModel()
# dm = extended_model.decmem

# extended_facts = [
#     actr.makechunk(typename="countOrder", first=1, second=2),
#     # ... add more facts here ...
# ]

### Exercise 2: Count Backwards

Create a model that counts backwards from 5 to 1. You'll need a different chunk type and different productions.

In [ ]:
# Your code here

# actr.chunktype("backwardOrder", ("first", "second"))
# backward_model = actr.ACTRModel()

### Exercise 3: Model Counting Errors

Add noise to the model to simulate occasional counting errors.

In [ ]:
# Your code here

# noisy_model = actr.ACTRModel()
# noisy_model.set_parameter("activation_noise", 0.5)

### Exercise 4: Cognitive Load

Create two models: one that just counts, and one that counts while doing another task. How does dual-tasking affect performance?

In [ ]:
# Your code here

## Summary

You've built your first ACT-R model and learned the core concepts: chunks, buffers, productions, and modules. You've seen how models help us understand cognition by forcing us to specify exact cognitive mechanisms, and how timing parameters allow us to model different people or conditions.

Each ACT-R model is a precise theory of how a cognitive task is performed. We can compare model behavior to human data to test our theories.

Next up: more complex chunk structures and declarative memory dynamics.

---

## Additional Resources

- [ACT-R Official Website](http://act-r.psy.cmu.edu/)
- [pyactr Documentation](https://github.com/jakdot/pyactr)
- [ACT-R Tutorial](http://act-r.psy.cmu.edu/actr7/tutorial/)

Research papers:
- Anderson, J. R. (2007). How Can the Human Mind Occur in the Physical Universe?
- Taatgen, N. A. (2013). The nature and transfer of cognitive skills.